# 🛡️ PortfolioIQ — Portfolio Risk Scorer

**Goal:** Given a client portfolio's sector allocations, compute a **risk score (0–100)** with risk category, top risk drivers, and rebalancing suggestions.

**Approach:**
1. Define realistic sector volatilities and correlation matrix from historical data
2. Generate thousands of synthetic portfolios (random + archetypal allocations)
3. Compute ground-truth risk scores using financial formulas (portfolio variance, HHI concentration, correlation penalty)
4. Train XGBoost regressor on portfolio features → risk score
5. Derive risk drivers analytically (decompose which factors contribute most)
6. Export for SageMaker deployment

**How this connects to Model 1 (Event Impact Classifier):**
- Model 1 says: "This news impacts energy +12%, tech -4%"
- Model 2 says: "This portfolio has risk score 73 because it's 60% energy → HIGH exposure to that event"
- Together: advisors see both the event AND how much it hurts their specific client

**Sectors:** tech, financials, energy, healthcare, bonds, commodities, international

---

## 0. Install Dependencies

In [ ]:
!pip install xgboost scikit-learn pandas numpy joblib matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import json
import random
from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports ready.")

---
## 1. Sector Market Data

Historical volatilities and correlations between the 7 sectors.
These approximate real ETF data (XLK, XLF, XLE, XLV, AGG, DJP, VEA).

| Sector | ETF Proxy | Annualized Vol |
|--------|-----------|----------------|
| Tech | XLK | 22% |
| Financials | XLF | 20% |
| Energy | XLE | 28% |
| Healthcare | XLV | 16% |
| Bonds | AGG | 6% |
| Commodities | DJP | 18% |
| International | VEA | 18% |

In [ ]:
SECTORS = ["tech", "financials", "energy", "healthcare", "bonds", "commodities", "international"]
N_SECTORS = len(SECTORS)

# ── Annualized volatilities (std dev of returns) ──────────────────
SECTOR_VOLS = {
    "tech":          0.22,
    "financials":    0.20,
    "energy":        0.28,
    "healthcare":    0.16,
    "bonds":         0.06,
    "commodities":   0.18,
    "international": 0.18,
}

# ── Correlation matrix (symmetric, approximate from historical ETF data) ──
# Order: tech, financials, energy, healthcare, bonds, commodities, international
CORR_MATRIX = np.array([
    # tech   fin    ener   health bonds  comm   intl
    [ 1.00,  0.70,  0.30,  0.50, -0.20,  0.10,  0.60],  # tech
    [ 0.70,  1.00,  0.50,  0.40,  0.00,  0.20,  0.65],  # financials
    [ 0.30,  0.50,  1.00,  0.20, -0.10,  0.65,  0.45],  # energy
    [ 0.50,  0.40,  0.20,  1.00,  0.10,  0.05,  0.40],  # healthcare
    [-0.20,  0.00, -0.10,  0.10,  1.00, -0.10, -0.05],  # bonds
    [ 0.10,  0.20,  0.65,  0.05, -0.10,  1.00,  0.35],  # commodities
    [ 0.60,  0.65,  0.45,  0.40, -0.05,  0.35,  1.00],  # international
])

# Build covariance matrix: Σ = diag(σ) @ ρ @ diag(σ)
vol_array = np.array([SECTOR_VOLS[s] for s in SECTORS])
COV_MATRIX = np.outer(vol_array, vol_array) * CORR_MATRIX

print("Sector Volatilities:")
for s, v in SECTOR_VOLS.items():
    print(f"  {s:<15} {v*100:.0f}%")

print(f"\nCorrelation Matrix shape: {CORR_MATRIX.shape}")
print(f"Covariance Matrix shape:  {COV_MATRIX.shape}")

# Verify positive semi-definite
eigenvalues = np.linalg.eigvals(COV_MATRIX)
print(f"Min eigenvalue: {eigenvalues.min():.6f} (must be ≥ 0 for valid covariance matrix)")
print("✅ Valid covariance matrix" if eigenvalues.min() >= -1e-10 else "❌ Invalid covariance matrix!")

---
## 2. Risk Score Formula

The ground-truth risk score combines multiple financial risk measures:

1. **Portfolio Volatility** — $\sigma_p = \sqrt{w^T \Sigma w}$ — the classic Markowitz portfolio risk
2. **Concentration (HHI)** — $HHI = \sum w_i^2$ — penalizes putting all eggs in one basket
3. **Correlation Penalty** — weighted average correlation between held sectors
4. **Max Drawdown Estimate** — $MDD \approx \sigma_p \times 2.5$ — rough worst-case scenario
5. **Bond Hedge Bonus** — having bonds reduces tail risk

Final score: weighted combination, scaled to 0–100.

In [ ]:
def compute_portfolio_risk_score(weights: np.ndarray) -> dict:
    """
    Compute a comprehensive risk score (0-100) for a portfolio.

    Args:
        weights: array of 7 sector weights (must sum to ~1.0)

    Returns:
        dict with risk_score, risk_category, component scores, and risk_drivers
    """
    w = np.array(weights, dtype=np.float64)
    w = w / w.sum()  # normalize to exactly 1.0

    # ── 1. Portfolio Volatility (0-60 scale) — PRIMARY DRIVER ─────
    port_variance = w @ COV_MATRIX @ w
    port_vol = np.sqrt(port_variance)  # annualized std dev
    # Scale: 4% vol → 0, 28% vol → 60
    vol_score = np.clip((port_vol - 0.04) / (0.28 - 0.04) * 60, 0, 65)

    # ── 2. Concentration (HHI) — weighted by sector volatility ───
    # Concentrating in bonds (6% vol) is much less risky than energy (28%)
    hhi = np.sum(w ** 2)
    weighted_vol_ratio = np.sum(w * vol_array) / vol_array.max()  # 0-1 scale
    concentration_penalty = np.clip((hhi - 0.143) / (1.0 - 0.143) * 18 * weighted_vol_ratio, 0, 18)

    # ── 3. Weighted Average Correlation ──────────────────────────
    active_sectors = np.where(w > 0.05)[0]  # sectors with >5% weight
    if len(active_sectors) > 1:
        corr_sum = 0
        weight_sum = 0
        for i in range(len(active_sectors)):
            for j in range(i+1, len(active_sectors)):
                si, sj = active_sectors[i], active_sectors[j]
                pair_weight = w[si] * w[sj]
                corr_sum += CORR_MATRIX[si, sj] * pair_weight
                weight_sum += pair_weight
        avg_corr = corr_sum / weight_sum if weight_sum > 0 else 0
    else:
        avg_corr = 1.0  # single sector = perfectly correlated with itself
    corr_penalty = np.clip((avg_corr + 0.2) / 1.2 * 10, -2, 10)

    # ── 4. Max Drawdown Estimate ─────────────────────────────────
    est_max_drawdown = port_vol * 2.5  # rough approximation
    drawdown_penalty = np.clip((est_max_drawdown - 0.15) / 0.55 * 8, 0, 10)

    # ── 5. Bond Hedge Bonus ──────────────────────────────────────
    bond_weight = w[SECTORS.index("bonds")]
    bond_bonus = 0
    if bond_weight >= 0.10:
        bond_bonus = -min(bond_weight * 15, 6)  # up to -6 points

    # ── 6. Single-Sector Dominance (scaled by that sector's vol) ─
    max_weight = w.max()
    max_idx = int(np.argmax(w))
    dominance_penalty = 0
    if max_weight > 0.50:
        sector_vol_factor = vol_array[max_idx] / vol_array.max()  # bonds=0.21, energy=1.0
        dominance_penalty = (max_weight - 0.50) * 30 * sector_vol_factor

    # ── 7. No Bond Penalty — missing downside protection ────────
    no_bond_penalty = 0
    if bond_weight < 0.05 and port_vol > 0.12:
        no_bond_penalty = 3.0  # extra risk for volatile portfolios with no hedge

    # ── Combine (ADDITIVE — each component already scaled) ───────
    raw_score = (
        vol_score +              # 0-65: the core risk measure
        concentration_penalty +  # 0-18: penalizes concentration (vol-weighted)
        corr_penalty +           # -2 to 10: correlated holdings
        drawdown_penalty +       # 0-10: tail risk
        bond_bonus +             # -6 to 0: bonds reduce risk
        dominance_penalty +      # 0-15: single sector dominance (vol-weighted)
        no_bond_penalty          # 0-3: no bond hedge
    )

    risk_score = int(np.clip(raw_score, 0, 100))

    # ── Category ─────────────────────────────────────────────────
    if risk_score <= 25:
        category = "LOW"
    elif risk_score <= 50:
        category = "MODERATE"
    elif risk_score <= 75:
        category = "HIGH"
    else:
        category = "CRITICAL"

    # ── Risk Drivers (top 3 contributors) ────────────────────────
    driver_scores = {
        f"Portfolio volatility ({port_vol*100:.1f}% annualized)": vol_score,
        f"Concentration — HHI={hhi:.2f}, top sector at {max_weight*100:.0f}%": concentration_penalty + dominance_penalty,
        f"Sector correlation — avg ρ={avg_corr:.2f} across holdings": corr_penalty,
        f"Drawdown risk — est. max drawdown {est_max_drawdown*100:.0f}%": drawdown_penalty,
    }
    if bond_weight >= 0.10:
        driver_scores[f"Bond hedge — {bond_weight*100:.0f}% bonds reducing tail risk"] = bond_bonus
    elif bond_weight < 0.05:
        driver_scores["No bond hedge — missing downside protection"] = no_bond_penalty + 2.0

    sorted_drivers = sorted(driver_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    drivers = [d[0] for d in sorted_drivers[:3]]

    return {
        "risk_score": risk_score,
        "risk_category": category,
        "risk_drivers": drivers,
        "components": {
            "portfolio_vol": round(float(port_vol), 4),
            "hhi": round(float(hhi), 4),
            "avg_correlation": round(float(avg_corr), 4),
            "est_max_drawdown": round(float(est_max_drawdown), 4),
            "bond_weight": round(float(bond_weight), 4),
            "max_sector_weight": round(float(max_weight), 4),
            "n_active_sectors": int(len(active_sectors)),
        },
        "vol_score": round(float(vol_score), 2),
        "concentration_penalty": round(float(concentration_penalty), 2),
        "corr_penalty": round(float(corr_penalty), 2),
        "drawdown_penalty": round(float(drawdown_penalty), 2),
        "bond_bonus": round(float(bond_bonus), 2),
        "dominance_penalty": round(float(dominance_penalty), 2),
    }


# ── Quick sanity check with known portfolios ─────────────────────
test_portfolios = {
    "All Bonds (safest)": [0, 0, 0, 0, 1, 0, 0],
    "All Energy (riskiest)": [0, 0, 1, 0, 0, 0, 0],
    "Equal Weight": [1/7]*7,
    "60/40 (Tech/Bonds)": [0.60, 0, 0, 0, 0.40, 0, 0],
    "Aggressive Tech": [0.70, 0.10, 0, 0, 0.05, 0.05, 0.10],
    "Balanced": [0.20, 0.15, 0.10, 0.15, 0.20, 0.05, 0.15],
    "Oil + Commodities": [0, 0, 0.45, 0, 0, 0.45, 0.10],
}

print("═" * 70)
print("RISK SCORE SANITY CHECK")
print("═" * 70)
for name, weights in test_portfolios.items():
    result = compute_portfolio_risk_score(weights)
    print(f"\n{name}:")
    print(f"  Score: {result['risk_score']}/100  [{result['risk_category']}]")
    print(f"  Vol: {result['components']['portfolio_vol']*100:.1f}%  |  HHI: {result['components']['hhi']:.3f}  |  Corr: {result['components']['avg_correlation']:.2f}")
    print(f"  Top driver: {result['risk_drivers'][0]}")

---
## 3. Generate Synthetic Portfolio Dataset

We generate portfolios in several ways to cover the full risk spectrum:
- **Random Dirichlet** — realistic random allocations with varying concentration
- **Archetypal** — known portfolio types (aggressive, conservative, balanced, etc.)
- **Extreme** — edge cases (100% single sector, near-zero in one sector)
- **Noisy variations** — perturbed versions of archetypes

In [ ]:
def generate_portfolios(n_total=15000):
    """
    Generate diverse synthetic portfolios covering the full risk spectrum.
    """
    portfolios = []

    # ── 1. Random Dirichlet portfolios (bulk of the data) ─────────
    # Different alpha parameters control concentration:
    # alpha < 1 → concentrated, alpha = 1 → uniform random, alpha > 1 → diversified
    n_dirichlet = int(n_total * 0.55)
    for _ in range(n_dirichlet):
        # Randomly pick concentration level
        alpha = np.random.choice([0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0])
        weights = np.random.dirichlet(np.ones(N_SECTORS) * alpha)
        portfolios.append({"weights": weights, "type": "random_dirichlet"})

    # ── 2. Archetypal portfolios with noise ───────────────────────
    archetypes = {
        # name: (base_weights, count)
        "conservative":     ([0.05, 0.10, 0.05, 0.15, 0.45, 0.05, 0.15], 400),
        "moderate":         ([0.15, 0.15, 0.10, 0.15, 0.25, 0.05, 0.15], 400),
        "balanced":         ([0.20, 0.15, 0.10, 0.15, 0.20, 0.05, 0.15], 400),
        "growth":           ([0.30, 0.15, 0.10, 0.10, 0.15, 0.05, 0.15], 400),
        "aggressive":       ([0.40, 0.15, 0.10, 0.05, 0.10, 0.05, 0.15], 400),
        "tech_heavy":       ([0.55, 0.10, 0.05, 0.05, 0.10, 0.05, 0.10], 350),
        "income":           ([0.05, 0.20, 0.05, 0.10, 0.40, 0.05, 0.15], 350),
        "energy_play":      ([0.05, 0.05, 0.45, 0.05, 0.10, 0.20, 0.10], 300),
        "healthcare_focus": ([0.10, 0.10, 0.05, 0.40, 0.20, 0.05, 0.10], 300),
        "commodities_heavy":[0.05, 0.05, 0.15, 0.05, 0.10, 0.45, 0.15],
        "global":           ([0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.40], 300),
        "all_bonds":        ([0.02, 0.03, 0.02, 0.03, 0.80, 0.05, 0.05], 200),
        "risk_parity_ish":  ([0.12, 0.13, 0.10, 0.16, 0.30, 0.09, 0.10], 300),
    }

    n_archetypal = int(n_total * 0.30)
    for name, spec in archetypes.items():
        if isinstance(spec, tuple):
            base, count = spec
        else:
            base, count = spec, 250
        base = np.array(base, dtype=np.float64)
        base = base / base.sum()

        for _ in range(count):
            # Add noise: ±5% per sector, then renormalize
            noise = np.random.normal(0, 0.04, N_SECTORS)
            noisy_weights = np.clip(base + noise, 0.01, 0.95)
            noisy_weights = noisy_weights / noisy_weights.sum()
            portfolios.append({"weights": noisy_weights, "type": f"archetype_{name}"})

    # ── 3. Extreme / edge case portfolios ────────────────────────
    n_extreme = int(n_total * 0.10)
    for _ in range(n_extreme // 3):
        # Single-sector dominant (60-95% in one sector)
        dominant = random.randint(0, N_SECTORS - 1)
        dom_weight = random.uniform(0.60, 0.95)
        remaining = np.random.dirichlet(np.ones(N_SECTORS - 1)) * (1 - dom_weight)
        weights = np.insert(remaining, dominant, dom_weight)
        portfolios.append({"weights": weights, "type": "extreme_concentrated"})

    for _ in range(n_extreme // 3):
        # Two-sector barbell
        s1, s2 = random.sample(range(N_SECTORS), 2)
        split = random.uniform(0.3, 0.7)
        weights = np.full(N_SECTORS, 0.01)
        weights[s1] = split
        weights[s2] = 1 - split - 0.01 * (N_SECTORS - 2)
        weights = np.clip(weights, 0.005, 0.99)
        weights = weights / weights.sum()
        portfolios.append({"weights": weights, "type": "extreme_barbell"})

    for _ in range(n_extreme // 3):
        # Near-equal weight with slight tilts
        weights = np.ones(N_SECTORS) / N_SECTORS
        tilt = np.random.normal(0, 0.02, N_SECTORS)
        weights = np.clip(weights + tilt, 0.05, 0.30)
        weights = weights / weights.sum()
        portfolios.append({"weights": weights, "type": "extreme_equal"})

    # ── 4. Fill remaining with extra random ──────────────────────
    while len(portfolios) < n_total:
        alpha = random.uniform(0.2, 5.0)
        weights = np.random.dirichlet(np.ones(N_SECTORS) * alpha)
        portfolios.append({"weights": weights, "type": "random_fill"})

    return portfolios[:n_total]


# Generate!
portfolios = generate_portfolios(n_total=15000)
print(f"Generated {len(portfolios)} portfolios")

# Count by type
type_counts = defaultdict(int)
for p in portfolios:
    type_counts[p['type']] += 1
print(f"\nPortfolio type distribution:")
for ptype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ptype:<30} {count:>5}")

In [ ]:
# ── Compute risk scores for all portfolios ──────────────────────

rows = []
for p in portfolios:
    w = p["weights"]
    result = compute_portfolio_risk_score(w)

    row = {
        "portfolio_type": p["type"],
        # Raw weights as features
        **{f"w_{s}": round(float(w[i]), 6) for i, s in enumerate(SECTORS)},
        # Derived features
        "hhi": result["components"]["hhi"],
        "portfolio_vol": result["components"]["portfolio_vol"],
        "avg_correlation": result["components"]["avg_correlation"],
        "est_max_drawdown": result["components"]["est_max_drawdown"],
        "max_sector_weight": result["components"]["max_sector_weight"],
        "n_active_sectors": result["components"]["n_active_sectors"],
        "bond_weight": result["components"]["bond_weight"],
        # Target
        "risk_score": result["risk_score"],
        "risk_category": result["risk_category"],
    }
    rows.append(row)

df = pd.DataFrame(rows)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f"Dataset shape: {df.shape}")
print(f"\nRisk score distribution:")
print(df['risk_score'].describe().round(1))
print(f"\nRisk category distribution:")
print(df['risk_category'].value_counts())

In [ ]:
# ── Visualize risk score distribution ─────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Risk score histogram
axes[0].hist(df['risk_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Risk Score Distribution')
axes[0].axvline(25, color='green', linestyle='--', alpha=0.7, label='LOW/MOD')
axes[0].axvline(50, color='orange', linestyle='--', alpha=0.7, label='MOD/HIGH')
axes[0].axvline(75, color='red', linestyle='--', alpha=0.7, label='HIGH/CRIT')
axes[0].legend(fontsize=8)

# 2. Risk score vs portfolio vol
axes[1].scatter(df['portfolio_vol']*100, df['risk_score'], alpha=0.1, s=5, c='steelblue')
axes[1].set_xlabel('Portfolio Volatility (%)')
axes[1].set_ylabel('Risk Score')
axes[1].set_title('Risk Score vs Portfolio Volatility')

# 3. Risk score vs HHI
axes[2].scatter(df['hhi'], df['risk_score'], alpha=0.1, s=5, c='coral')
axes[2].set_xlabel('HHI (Concentration)')
axes[2].set_ylabel('Risk Score')
axes[2].set_title('Risk Score vs Concentration (HHI)')

plt.tight_layout()
plt.show()

print("Correlations with risk_score:")
feature_cols = [c for c in df.columns if c.startswith('w_') or c in ['hhi', 'portfolio_vol', 'avg_correlation', 'max_sector_weight', 'n_active_sectors', 'bond_weight']]
for col in feature_cols:
    corr = df[col].corr(df['risk_score'])
    print(f"  {col:<25} r={corr:>+.3f}")

---
## 4. Feature Engineering

The model gets:
- **7 sector weights** — the raw allocation percentages
- **6 derived features** — HHI, portfolio vol, avg correlation, max drawdown estimate, max sector weight, number of active sectors
- **Total: 13 features** → predict 1 target (risk score 0-100)

In [ ]:
# ── Define features and target ────────────────────────────────────

feature_cols = [
    # Raw weights
    "w_tech", "w_financials", "w_energy", "w_healthcare",
    "w_bonds", "w_commodities", "w_international",
    # Derived features
    "hhi", "portfolio_vol", "avg_correlation",
    "est_max_drawdown", "max_sector_weight", "n_active_sectors",
]

X = df[feature_cols].values
y = df["risk_score"].values

print(f"Feature matrix: {X.shape}")
print(f"Target vector:  {y.shape}")
print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols):
    print(f"  [{i:>2}] {col:<25} range: [{df[col].min():.4f}, {df[col].max():.4f}]")

In [ ]:
# ── Train/test split ──────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

---
## 5. Train XGBoost Risk Scorer

In [ ]:
# ── Train XGBoost regressor ──────────────────────────────────────

risk_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=3,
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30,
    eval_metric='mae',
)

print("Training XGBoost risk scorer...")
risk_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print(f"\nBest iteration: {risk_model.best_iteration}")
print("Training complete!")

---
## 6. Evaluation

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────

y_pred = risk_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("═" * 60)
print("TEST SET EVALUATION")
print("═" * 60)
print(f"\nMAE:   {mae:.2f} points  (avg error on 0-100 scale)")
print(f"RMSE:  {rmse:.2f} points")
print(f"R²:    {r2:.4f}")

# Category accuracy
def score_to_category(score):
    if score <= 25: return "LOW"
    elif score <= 50: return "MODERATE"
    elif score <= 75: return "HIGH"
    else: return "CRITICAL"

true_cats = [score_to_category(s) for s in y_test]
pred_cats = [score_to_category(s) for s in y_pred]
cat_acc = sum(1 for t, p in zip(true_cats, pred_cats) if t == p) / len(true_cats)
print(f"\nCategory accuracy: {cat_acc:.1%}")
print("(Does the model place portfolios in the correct risk bucket?)")

# Within ±5 points accuracy
within_5 = np.mean(np.abs(y_test - y_pred) <= 5)
within_10 = np.mean(np.abs(y_test - y_pred) <= 10)
print(f"\nWithin ±5 points:  {within_5:.1%}")
print(f"Within ±10 points: {within_10:.1%}")

print(f"\n✅ MAE < 3 points is excellent on a 0-100 scale.")
print(f"✅ R² > 0.95 means the model captures 95%+ of risk variation.")

In [ ]:
# ── Actual vs Predicted scatter plot ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.15, s=8, c='steelblue')
axes[0].plot([0, 100], [0, 100], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Risk Score')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.4f})')
axes[0].legend()
axes[0].set_xlim(-5, 105)
axes[0].set_ylim(-5, 105)

# Error distribution
errors = y_pred - y_test
axes[1].hist(errors, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Prediction Error (Predicted - Actual)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Error Distribution (MAE={mae:.2f})')
axes[1].axvline(0, color='black', linestyle='-', linewidth=1)

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance ────────────────────────────────────────────

importances = risk_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print("Feature Importance Ranking:")
print("-" * 45)
for rank, idx in enumerate(sorted_idx):
    bar = "█" * int(importances[idx] * 50)
    print(f"  {rank+1:>2}. {feature_cols[idx]:<25} {importances[idx]:.4f} {bar}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    [feature_cols[i] for i in sorted_idx[::-1]],
    [importances[i] for i in sorted_idx[::-1]],
    color='steelblue', edgecolor='white'
)
ax.set_xlabel('Feature Importance')
ax.set_title('XGBoost Feature Importance for Risk Scoring')
plt.tight_layout()
plt.show()

---
## 7. Test on Known Portfolio Archetypes

Does the model rank these portfolios in the correct order of risk?

In [ ]:
# ── Test on specific portfolio types ──────────────────────────────

def predict_risk(weights: list) -> dict:
    """
    Predict risk score for a portfolio using the ML model.
    Also computes analytical risk drivers.

    Args:
        weights: list of 7 sector weights (must sum to ~1.0)

    Returns:
        dict with risk_score, risk_category, risk_drivers, suggested_rebalance
    """
    w = np.array(weights, dtype=np.float64)
    w = w / w.sum()

    # Compute derived features
    analytical = compute_portfolio_risk_score(w)
    components = analytical["components"]

    # Build feature vector (same order as training)
    features = np.array([
        *w,  # 7 sector weights
        components["hhi"],
        components["portfolio_vol"],
        components["avg_correlation"],
        components["est_max_drawdown"],
        components["max_sector_weight"],
        components["n_active_sectors"],
    ]).reshape(1, -1)

    # ML prediction
    ml_score = float(np.clip(risk_model.predict(features)[0], 0, 100))

    # Category
    category = score_to_category(ml_score)

    # ── Suggested rebalancing ────────────────────────────────────
    suggestions = []
    max_idx = np.argmax(w)
    max_sector = SECTORS[max_idx]
    bond_idx = SECTORS.index("bonds")

    if w[max_idx] > 0.40:
        suggestions.append(f"Reduce {max_sector} from {w[max_idx]*100:.0f}% to ≤35%")

    if w[bond_idx] < 0.10 and ml_score > 40:
        suggestions.append(f"Increase bonds from {w[bond_idx]*100:.0f}% to ≥15% for downside protection")

    # Check for correlated pair overweight
    high_corr_pairs = [
        ("tech", "financials", 0.70),
        ("energy", "commodities", 0.65),
        ("tech", "international", 0.60),
        ("financials", "international", 0.65),
    ]
    for s1, s2, corr in high_corr_pairs:
        i1, i2 = SECTORS.index(s1), SECTORS.index(s2)
        combined = w[i1] + w[i2]
        if combined > 0.50:
            suggestions.append(f"Reduce {s1}+{s2} combined ({combined*100:.0f}%) — correlation {corr:.2f}")

    n_active = sum(1 for x in w if x > 0.05)
    if n_active < 4:
        suggestions.append(f"Diversify — only {n_active} sectors above 5%")

    if not suggestions:
        suggestions.append("Portfolio is well-balanced — no immediate action needed")

    return {
        "risk_score": round(ml_score, 1),
        "risk_category": category,
        "risk_drivers": analytical["risk_drivers"],
        "suggested_rebalance": suggestions[:3],
        "allocations": {s: round(float(w[i])*100, 1) for i, s in enumerate(SECTORS)},
        "components": components,
        "analytical_score": analytical["risk_score"],  # for comparison
    }


# ── Test portfolios ───────────────────────────────────────────────
test_cases = [
    ("Retiree — Conservative",       [0.05, 0.10, 0.05, 0.10, 0.55, 0.05, 0.10]),
    ("Young Professional — Growth",   [0.35, 0.15, 0.10, 0.10, 0.10, 0.05, 0.15]),
    ("Tech Bro — All In FAANG",       [0.75, 0.05, 0.02, 0.03, 0.05, 0.02, 0.08]),
    ("Balanced 60/40 Traditional",     [0.18, 0.12, 0.10, 0.10, 0.40, 0.02, 0.08]),
    ("Energy Bull — Oil & Gas",        [0.05, 0.05, 0.55, 0.05, 0.05, 0.15, 0.10]),
    ("Risk Parity Approach",           [0.12, 0.13, 0.09, 0.16, 0.32, 0.08, 0.10]),
    ("Doomsday Prepper — Gold & Bonds",[0.02, 0.03, 0.05, 0.05, 0.40, 0.40, 0.05]),
    ("YOLO Gambler — Energy+Crypto",   [0.05, 0.02, 0.45, 0.02, 0.01, 0.40, 0.05]),
]

print("═" * 90)
print("PORTFOLIO RISK ASSESSMENT — ML MODEL vs ANALYTICAL FORMULA")
print("═" * 90)

for name, weights in test_cases:
    result = predict_risk(weights)
    emoji = {"LOW": "🟢", "MODERATE": "🟡", "HIGH": "🟠", "CRITICAL": "🔴"}[result['risk_category']]

    print(f"\n{emoji} {name}")
    print(f"   ML Score: {result['risk_score']}/100  [{result['risk_category']}]  |  Analytical: {result['analytical_score']}/100")
    print(f"   Vol: {result['components']['portfolio_vol']*100:.1f}%  |  HHI: {result['components']['hhi']:.3f}  |  MaxSector: {result['components']['max_sector_weight']*100:.0f}%")
    print(f"   Top driver: {result['risk_drivers'][0]}")
    print(f"   💡 {result['suggested_rebalance'][0]}")

---
## 8. Risk Category Classifier (bonus)

Train a separate model to directly classify into LOW/MODERATE/HIGH/CRITICAL.
Useful when you need a hard category without relying on score thresholds.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# ── Encode categories ─────────────────────────────────────────────
CATEGORY_LABELS = ["LOW", "MODERATE", "HIGH", "CRITICAL"]
cat_to_int = {c: i for i, c in enumerate(CATEGORY_LABELS)}
ALL_LABELS = list(range(len(CATEGORY_LABELS)))

y_cat = np.array([cat_to_int[score_to_category(s)] for s in y])
y_cat_train = np.array([cat_to_int[score_to_category(s)] for s in y_train])
y_cat_test = np.array([cat_to_int[score_to_category(s)] for s in y_test])

print(f"Category distribution in test set:")
for i, label in enumerate(CATEGORY_LABELS):
    count = np.sum(y_cat_test == i)
    print(f"  {label}: {count}")

# ── Train classifier ─────────────────────────────────────────────
cat_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    num_class=4,
    objective='multi:softmax',
    eval_metric='mlogloss',
)

print("\nTraining risk category classifier...")
cat_model.fit(X_train, y_cat_train)

y_cat_pred = cat_model.predict(X_test)
cat_accuracy = np.mean(y_cat_pred == y_cat_test)

print(f"\nCategory Classification Accuracy: {cat_accuracy:.1%}")
print(f"\nClassification Report:")
print(classification_report(y_cat_test, y_cat_pred, labels=ALL_LABELS, target_names=CATEGORY_LABELS, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_cat_test, y_cat_pred, labels=ALL_LABELS)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORY_LABELS, yticklabels=CATEGORY_LABELS, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Risk Category Confusion Matrix (Accuracy: {cat_accuracy:.1%})')
plt.tight_layout()
plt.show()

---
## 9. Event Exposure Integration

This is where **Model 1 (Event Impact Classifier) connects to Model 2 (Risk Scorer)**.

Given current news events and their sector impacts, compute how exposed a portfolio is to those events.

**Event Exposure** = $\sum_{i} w_i \times |\text{event\_impact}_i|$

High event exposure + high base risk = advisor should act NOW.

In [ ]:
def compute_event_exposure(weights: list, event_impacts: dict) -> dict:
    """
    Compute how exposed a portfolio is to a specific news event.

    Args:
        weights: list of 7 sector weights
        event_impacts: dict of {sector: impact_pct} from Event Impact Classifier

    Returns:
        dict with exposure_score, affected_holdings, estimated_pnl
    """
    w = np.array(weights, dtype=np.float64)
    w = w / w.sum()

    # Weighted impact on this portfolio
    portfolio_impact = 0.0
    affected = []

    for i, sector in enumerate(SECTORS):
        sector_impact = event_impacts.get(sector, 0.0)
        weighted_impact = w[i] * sector_impact
        portfolio_impact += weighted_impact

        if abs(sector_impact) > 0.01 and w[i] > 0.05:
            affected.append({
                "sector": sector,
                "allocation": round(float(w[i]) * 100, 1),
                "event_impact": round(float(sector_impact) * 100, 2),
                "weighted_impact": round(float(weighted_impact) * 100, 2),
            })

    affected.sort(key=lambda x: abs(x["weighted_impact"]), reverse=True)

    # Exposure score: how much of the portfolio is touched by this event
    exposure = sum(w[i] * abs(event_impacts.get(s, 0)) for i, s in enumerate(SECTORS))

    if exposure > 0.08:
        urgency = "URGENT"
    elif exposure > 0.04:
        urgency = "MONITOR"
    else:
        urgency = "LOW"

    return {
        "portfolio_impact_pct": round(float(portfolio_impact) * 100, 2),
        "exposure_score": round(float(exposure) * 100, 2),
        "urgency": urgency,
        "affected_holdings": affected[:5],
    }


# ── Demo: combine Event Impact Classifier output with Risk Scorer ──

# Simulated event: "Fed raises rates by 100bps" → impacts from Model 1
event_impacts = {
    "bonds": -0.10,
    "tech": -0.12,
    "financials": +0.05,
    "commodities": -0.04,
}

demo_portfolios = [
    ("Tech-Heavy Portfolio",     [0.55, 0.10, 0.05, 0.05, 0.10, 0.05, 0.10]),
    ("Bond-Heavy Conservative",  [0.05, 0.10, 0.05, 0.10, 0.55, 0.05, 0.10]),
    ("Balanced Portfolio",        [0.18, 0.14, 0.10, 0.14, 0.25, 0.05, 0.14]),
    ("Bank/Financials Overweight",[0.10, 0.45, 0.05, 0.10, 0.15, 0.05, 0.10]),
]

print("═" * 85)
print('EVENT: "Fed raises rates by 100bps in emergency session"')
print(f"Event Impacts: {event_impacts}")
print("═" * 85)

for name, weights in demo_portfolios:
    risk = predict_risk(weights)
    exposure = compute_event_exposure(weights, event_impacts)

    urgency_emoji = {"URGENT": "🚨", "MONITOR": "⚠️", "LOW": "✅"}[exposure['urgency']]

    print(f"\n{urgency_emoji} {name}")
    print(f"   Base Risk: {risk['risk_score']}/100 [{risk['risk_category']}]")
    print(f"   Event Exposure: {exposure['exposure_score']}%  |  Portfolio Impact: {exposure['portfolio_impact_pct']:+.2f}%  |  Urgency: {exposure['urgency']}")
    if exposure['affected_holdings']:
        print(f"   Most affected:")
        for h in exposure['affected_holdings'][:3]:
            direction = "📈" if h['weighted_impact'] > 0 else "📉"
            print(f"     {direction} {h['sector']} ({h['allocation']}% held) → event: {h['event_impact']:+.2f}% → portfolio: {h['weighted_impact']:+.2f}%")

---
## 10. Export Model Artifacts

Save everything needed for SageMaker deployment:
- XGBoost risk score model
- XGBoost risk category classifier
- Sector data (vols, correlations)
- Metadata

In [ ]:
import os

EXPORT_DIR = "model_artifacts_risk"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save risk score model
joblib.dump(risk_model, f"{EXPORT_DIR}/risk_score_model.joblib")

# Save category classifier
joblib.dump(cat_model, f"{EXPORT_DIR}/risk_category_model.joblib")

# Save sector data (needed for feature computation at inference time)
sector_data = {
    "sectors": SECTORS,
    "volatilities": SECTOR_VOLS,
    "correlation_matrix": CORR_MATRIX.tolist(),
    "covariance_matrix": COV_MATRIX.tolist(),
}
with open(f"{EXPORT_DIR}/sector_data.json", "w") as f:
    json.dump(sector_data, f, indent=2)

# Save metadata
metadata = {
    "model_type": "XGBoost Regressor + Classifier",
    "features": feature_cols,
    "n_features": len(feature_cols),
    "training_samples": int(X_train.shape[0]),
    "test_samples": int(X_test.shape[0]),
    "test_mae": round(float(mae), 2),
    "test_r2": round(float(r2), 4),
    "category_accuracy": round(float(cat_accuracy), 4),
    "risk_categories": CATEGORY_LABELS,
    "category_thresholds": {"LOW": "0-25", "MODERATE": "26-50", "HIGH": "51-75", "CRITICAL": "76-100"},
    "input_format": "13 features: 7 sector weights + 6 derived metrics (HHI, vol, corr, drawdown, max_weight, n_active)",
    "output_format": "risk_score (0-100), risk_category (LOW/MODERATE/HIGH/CRITICAL)",
    "inference_steps": [
        "1. Receive 7 sector weights from portfolio",
        "2. Compute derived features: HHI, portfolio vol, avg correlation, est max drawdown, max sector weight, n active sectors",
        "3. Stack into 13-dim feature vector",
        "4. Pass to risk_score_model.predict() → score 0-100",
        "5. Pass to risk_category_model.predict() → category index 0-3",
        "6. Compute risk drivers analytically from portfolio composition",
        "7. Generate rebalancing suggestions based on risk factors",
    ],
}

with open(f"{EXPORT_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved artifacts:")
for fname in sorted(os.listdir(EXPORT_DIR)):
    fsize = os.path.getsize(f"{EXPORT_DIR}/{fname}")
    print(f"  {fname:<35} {fsize / 1024:.1f} KB")

print(f"\n✅ All artifacts saved to '{EXPORT_DIR}/'")
print("📦 For SageMaker: upload these files — no GPU needed, pure XGBoost on tabular data")
print("   Much lighter than Model 1 (no sentence-transformers dependency)")

---
## 11. Interactive Playground 🎮

Edit the portfolio weights below and re-run to see the risk assessment!

In [ ]:
# ── Full inference function (same as SageMaker endpoint) ──────────

def assess_portfolio_risk(allocations: dict, event_impacts: dict = None) -> dict:
    """
    Full portfolio risk assessment — the exact function for the SageMaker endpoint.

    Args:
        allocations: dict of {sector: weight} e.g. {"tech": 0.40, "bonds": 0.30, ...}
        event_impacts: optional dict of {sector: impact} from Event Impact Classifier

    Returns:
        Complete risk assessment dict
    """
    # Build weight vector
    weights = [allocations.get(s, 0.0) for s in SECTORS]
    total = sum(weights)
    if total == 0:
        return {"error": "All weights are zero"}
    weights = [w / total for w in weights]  # normalize

    # Get risk score
    risk = predict_risk(weights)

    # Get event exposure if provided
    exposure = None
    if event_impacts:
        exposure = compute_event_exposure(weights, event_impacts)

    return {
        "risk_score": risk["risk_score"],
        "risk_category": risk["risk_category"],
        "risk_drivers": risk["risk_drivers"],
        "suggested_rebalance": risk["suggested_rebalance"],
        "allocations": risk["allocations"],
        "portfolio_metrics": risk["components"],
        "event_exposure": exposure,
    }


# ── Try it! ───────────────────────────────────────────────────────

# Edit this portfolio and re-run!
my_portfolio = {
    "tech":          0.35,
    "financials":    0.15,
    "energy":        0.10,
    "healthcare":    0.10,
    "bonds":         0.15,
    "commodities":   0.05,
    "international": 0.10,
}

# Optional: simulate a current news event
current_event = {
    "tech": -0.08,
    "financials": -0.04,
    "energy": +0.12,
    "commodities": +0.08,
    "international": -0.05,
}

result = assess_portfolio_risk(my_portfolio, current_event)

print("═" * 70)
print("PORTFOLIO RISK ASSESSMENT")
print("═" * 70)
print(f"\n🎯 Risk Score: {result['risk_score']}/100  [{result['risk_category']}]")
print(f"\n📊 Allocations:")
for sector, pct in result['allocations'].items():
    bar = "█" * int(pct / 2)
    print(f"   {sector:<15} {pct:>5.1f}% {bar}")

print(f"\n📈 Portfolio Metrics:")
m = result['portfolio_metrics']
print(f"   Volatility:      {m['portfolio_vol']*100:.1f}%")
print(f"   HHI:             {m['hhi']:.3f}")
print(f"   Avg Correlation: {m['avg_correlation']:.2f}")
print(f"   Est Max Drawdown:{m['est_max_drawdown']*100:.0f}%")

print(f"\n⚠️ Risk Drivers:")
for i, driver in enumerate(result['risk_drivers']):
    print(f"   {i+1}. {driver}")

print(f"\n💡 Suggestions:")
for s in result['suggested_rebalance']:
    print(f"   → {s}")

if result['event_exposure']:
    exp = result['event_exposure']
    print(f"\n🔥 Event Exposure:")
    print(f"   Portfolio Impact: {exp['portfolio_impact_pct']:+.2f}%")
    print(f"   Exposure Score:   {exp['exposure_score']}%")
    print(f"   Urgency:          {exp['urgency']}")

print(f"\n📦 JSON output (what Node.js backend receives):")
print(json.dumps(result, indent=2))

---
## ✅ Done!

### What you have now:
1. **Risk Score Model** — XGBoost regressor that predicts risk score 0–100 from portfolio allocations
2. **Risk Category Classifier** — XGBoost classifier for LOW/MODERATE/HIGH/CRITICAL
3. **Risk Driver Analysis** — analytical decomposition of why the score is what it is
4. **Event Exposure Engine** — connects Model 1 (Event Impact) to Model 2 (Risk Score)
5. **Rebalancing Suggestions** — actionable advice for advisors
6. **Exported artifacts** ready for SageMaker deployment

### How the two models work together:
```
News headline → Model 1 (Event Impact) → sector impacts
                                              ↓
Client portfolio → Model 2 (Risk Score) → risk_score + event_exposure
                                              ↓
                                    Dashboard: "Client X has risk score 73,
                                    HIGH exposure to Fed rate hike event.
                                    Suggested: reduce tech from 55% to 35%"
```

### Next steps:
1. **Test in Colab** — run all cells, verify risk ordering makes sense
2. **Deploy** — upload `model_artifacts_risk/` to S3 → SageMaker (lightweight, no GPU needed)
3. **Integrate** — Node.js backend calls both endpoints, combines results for dashboard
4. **Real data** — fine-tune with actual portfolio returns and risk events
5. **Next model** — Volatility Forecasting (Model 3)